In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm

cwd = Path.cwd()

if (cwd / "requirements.txt").exists():
    ROOT = cwd
elif (cwd.parent / "requirements.txt").exists():
    ROOT = cwd.parent
else:
    raise FileNotFoundError("Не найден корень проекта")

PROCESSED = ROOT / "data" / "processed"
TABLES = ROOT / "reports" / "tables"
TABLES.mkdir(parents=True, exist_ok=True)

buckets = pd.read_csv(
    PROCESSED / "btcusdt_tr_vpin_with_u2_2020_q1.csv"
)

buckets["available_at"] = pd.to_datetime(
    buckets["available_at"], utc=True
)

buckets = buckets.sort_values("bucket_id").reset_index(drop=True)

assert buckets["bucket_id"].is_unique
assert buckets["bucket_id"].diff().dropna().eq(1).all()

train_base = pd.read_csv(PROCESSED / "forecast_train.csv")
test_base = pd.read_csv(PROCESSED / "forecast_test.csv")

train_base["sample"] = "train"
test_base["sample"] = "test"

base = pd.concat([train_base, test_base], ignore_index=True)

for column in ["timestamp", "target_end"]:
    base[column] = pd.to_datetime(base[column], utc=True)

base = base.sort_values("timestamp").reset_index(drop=True)

print("Исходных прогнозных строк:", len(base))

Исходных прогнозных строк: 2118


In [2]:
WINDOWS = [25, 50, 100]
signal_columns = []

for window in WINDOWS:
    tr_name = f"tr_L{window}"
    u2_name = f"u2_L{window}"

    buckets[tr_name] = (
        buckets["oi"]
        .rolling(window, min_periods=window)
        .mean()
    )

    buckets[u2_name] = (
        buckets["u2"]
        .rolling(window, min_periods=window)
        .mean()
    )

    signal_columns.extend([tr_name, u2_name])

signals = (
    buckets
    .sort_values(["available_at", "bucket_id"])
    .drop_duplicates("available_at", keep="last")
    [["available_at", *signal_columns]]
    .rename(columns={"available_at": "variant_available_at"})
)

panel = pd.merge_asof(
    base,
    signals.sort_values("variant_available_at"),
    left_on="timestamp",
    right_on="variant_available_at",
    direction="backward",
    allow_exact_matches=True,
)

matched = panel["variant_available_at"].notna()

assert (
    panel.loc[matched, "variant_available_at"]
    <= panel.loc[matched, "timestamp"]
).all()

assert np.allclose(
    panel["tr_L50"],
    panel["tr_vpin"],
    equal_nan=True,
    rtol=1e-10,
    atol=1e-10,
)

assert np.allclose(
    panel["u2_L50"],
    panel["u2_vpin"],
    equal_nan=True,
    rtol=1e-10,
    atol=1e-10,
)

common = panel.dropna(subset=signal_columns).copy()

train = common.loc[common["sample"].eq("train")].copy()
test = common.loc[common["sample"].eq("test")].copy()

assert not train.empty
assert not test.empty
assert train["target_end"].max() <= test["timestamp"].min()

print("Проверка воспроизведения L=50: OK")
print("Общая обучающая выборка:", len(train))
print("Общая тестовая выборка:", len(test))
print("Исключено из-за прогрева окон:", len(panel) - len(common))

Проверка воспроизведения L=50: OK
Общая обучающая выборка: 1361
Общая тестовая выборка: 740
Исключено из-за прогрева окон: 17


In [3]:
TARGET = "rv_next_1h_pct"

controls = [
    "rv_prev_1h_pct",
    "log_volume_prev_1h",
]

specifications = {
    "Controls": controls,
}

for window in WINDOWS:
    specifications[f"Controls_TR_L{window}"] = [
        *controls, f"tr_L{window}"
    ]

    specifications[f"Controls_U2_L{window}"] = [
        *controls, f"u2_L{window}"
    ]

actual = test[TARGET].to_numpy(dtype=float)
rows = []

for name, features in specifications.items():
    X_train = sm.add_constant(
        train[features].astype(float),
        has_constant="add",
    )

    X_test = sm.add_constant(
        test[features].astype(float),
        has_constant="add",
    ).reindex(columns=X_train.columns)

    model = sm.OLS(
        train[TARGET].astype(float),
        X_train,
        missing="raise",
    ).fit()

    forecast = np.asarray(model.predict(X_test))
    error = forecast - actual

    rows.append({
        "model": name,
        "n_train": len(train),
        "n_test": len(test),
        "MSE": float(np.mean(error ** 2)),
        "RMSE": float(np.sqrt(np.mean(error ** 2))),
        "MAE": float(np.mean(np.abs(error))),
        "bias": float(np.mean(error)),
    })

persistence_error = (
    test["rv_prev_1h_pct"].to_numpy(dtype=float) - actual
)

rows.append({
    "model": "Persistence",
    "n_train": len(train),
    "n_test": len(test),
    "MSE": float(np.mean(persistence_error ** 2)),
    "RMSE": float(np.sqrt(np.mean(persistence_error ** 2))),
    "MAE": float(np.mean(np.abs(persistence_error))),
    "bias": float(np.mean(persistence_error)),
})

window_metrics = pd.DataFrame(rows).set_index("model")

reference_mse = window_metrics.loc["Controls", "MSE"]

window_metrics["MSE_gain_vs_controls_pct"] = (
    100 * (1 - window_metrics["MSE"] / reference_mse)
)

print(
    window_metrics
    .sort_values("RMSE")
    .round(6)
    .to_string()
)

window_metrics.to_csv(
    TABLES / "vpin_window_robustness.csv"
)

                  n_train  n_test       MSE      RMSE       MAE      bias  MSE_gain_vs_controls_pct
model                                                                                              
Persistence          1361     740  1.423754  1.193212  0.487237  0.000281                 18.289002
Controls_U2_L100     1361     740  1.670814  1.292600  0.534438 -0.451774                  4.109891
Controls_TR_L100     1361     740  1.693699  1.301422  0.543918 -0.467180                  2.796518
Controls_U2_L50      1361     740  1.712479  1.308617  0.552514 -0.474192                  1.718698
Controls_TR_L50      1361     740  1.731604  1.315904  0.560959 -0.486483                  0.621088
Controls_U2_L25      1361     740  1.740054  1.319111  0.564527 -0.488890                  0.136125
Controls             1361     740  1.742426  1.320010  0.566373 -0.491475                  0.000000
Controls_TR_L25      1361     740  1.745280  1.321090  0.566922 -0.492139                 -0.163781
